# LyCon reconstruction on Colab (a few hundred songs, Qwen2.5, 4-bit)

Runs the whole pipeline on a **free T4**: build prompts → generate with Qwen2.5 (4-bit) → compare against the authors' GPT-4o LyCon on the **same songs**.

Provide two files the notebook can't fetch for you — musiXmatch `mxm_dataset_train.txt` and AllMusic `msd-MASD-styleAssignment.cls` — by uploading them to your Drive.

## 1. Setup

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes
from google.colab import drive; drive.mount('/content/drive')
import os
WORK='/content/drive/MyDrive/lycon'   # outputs persist here across sessions
os.makedirs(WORK, exist_ok=True); os.chdir('/content')

## 2. Get code + public data
Clones the Deezer mood dataset and the authors' released LyCon set (matched reference). Upload this project's `src/` folder to `/content/drive/MyDrive/lycon/src` first (or clone your own repo).

In [ ]:
!git clone -q https://github.com/deezer/deezer_mood_detection_dataset
!git clone -q https://github.com/havenpersona/lycon
!cd lycon && unzip -q -o dataset.zip
import shutil, os
shutil.copytree('/content/drive/MyDrive/lycon/src', '/content/src', dirs_exist_ok=True)
os.chdir('/content')

## 3. Point to your uploaded mxm + AllMusic files

In [ ]:
MXM    = '/content/drive/MyDrive/lycon/mxm_dataset_train.txt'
STYLES = '/content/drive/MyDrive/lycon/msd-MASD-styleAssignment.cls'
import os; assert os.path.exists(MXM) and os.path.exists(STYLES), 'upload mxm + styles to Drive first'

## 4. Build a few-hundred-song subset (matched to the released set)

In [ ]:
!python /content/src/build_prompts.py \
  --deezer-dir /content/deezer_mood_detection_dataset \
  --mxm $MXM --styles $STYLES \
  --released-dir /content/lycon/dataset \
  --limit 300 --seed 13 \
  --out /content/drive/MyDrive/lycon/prompts.jsonl

## 5. Generate with Qwen2.5 (4-bit). `--resume` makes it restartable if the session drops.

In [ ]:
!python /content/src/generate.py \
  --manifest /content/drive/MyDrive/lycon/prompts.jsonl \
  --out-dir /content/drive/MyDrive/lycon/qwen_reconstructions \
  --model Qwen/Qwen2.5-7B-Instruct --load-4bit --resume

## 6. Matched comparison: Qwen vs the authors' GPT-4o, same songs
**Unique n-gram totals scale with corpus size**, so on a few hundred songs compare the per-song averages (word/line/section) and abstract/concrete ratios — not the absolute n-gram counts.

In [ ]:
!python /content/src/pull_reference.py --manifest /content/drive/MyDrive/lycon/prompts.jsonl \
  --released-dir /content/lycon/dataset --out-dir /content/drive/MyDrive/lycon/reference_subset
print('=== Authors GPT-4o LyCon (same songs) ===')
!python /content/src/lycon_stats.py --dir /content/drive/MyDrive/lycon/reference_subset
print('=== Your Qwen2.5 LyCon ===')
!python /content/src/lycon_stats.py --dir /content/drive/MyDrive/lycon/qwen_reconstructions